In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [11]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Exploring games data

In [57]:
clean_games_fp = Path(config['games_folder'] + config['games_clean_fp'])

games_df = pd.read_json(clean_games_fp, orient='records')
games_df['updated_at'] = pd.to_datetime(games_df['updated_at'], errors='coerce')
games_df['first_release_date'] = pd.to_datetime(games_df['first_release_date'], errors='coerce')

games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,The Final Station,74.095796,2026-04-29 20:19:42,2016-08-30,16136,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
1,Monster Jam Showdown,-1.000000,2026-04-29 20:19:38,2024-08-29,289698,0,1,0,1,1,...,0,0,0,1,1,0,0,0,0,0
2,Gentlemen Dispute,-1.000000,2026-04-29 20:19:13,2013-11-20,136086,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,S.K.I.L.L.: Special Force 2,62.748354,2026-04-29 20:18:25,2013-09-12,17174,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Elebits,63.502121,2026-04-29 20:18:23,2006-12-02,2747,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [58]:
games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,The Final Station,74.095796,2026-04-29 20:19:42,2016-08-30,16136,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
1,Monster Jam Showdown,-1.000000,2026-04-29 20:19:38,2024-08-29,289698,0,1,0,1,1,...,0,0,0,1,1,0,0,0,0,0
2,Gentlemen Dispute,-1.000000,2026-04-29 20:19:13,2013-11-20,136086,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,S.K.I.L.L.: Special Force 2,62.748354,2026-04-29 20:18:25,2013-09-12,17174,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,Elebits,63.502121,2026-04-29 20:18:23,2006-12-02,2747,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [59]:
#Checking to make sure the id uniquely identifies each row
print("No duplicate IDS") if len(games_df['id'].unique()) == len(games_df) else print("Duplicate IDs found")

No duplicate IDS


In [60]:
#Checking to see if duplicate game names exist
print("No duplicate game names") if len(games_df['name'].unique()) == len(games_df) else print("Duplicate game names found")

Duplicate game names found


In [61]:
#Get all rows with duplicate game names
duplicate_names_df = games_df[games_df.duplicated(subset=['name'], keep=False)].sort_values('name')
duplicate_names_df.head(6)

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
17016,10-Pin Bowling,-1.000000,2024-11-14 09:33:09,1984-12-31,153453,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
17017,10-Pin Bowling,-1.000000,2024-11-14 09:27:27,1999-08-01,92273,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
7119,15 Minutes,-1.000000,2026-04-09 19:38:22,2026-01-05,395433,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
12920,15 Minutes,-1.000000,2026-02-02 16:52:35,2025-10-23,355071,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5578,1942,61.392350,2026-04-18 08:37:59,1985-12-11,272544,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
6495,1942,67.742592,2026-04-13 08:03:19,1984-12-01,6075,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0


In [62]:
#Find columns where duplicates differ (excluding name, rating, updated_at, id)
exclude_cols = {'name', 'rating', 'updated_at', 'id'}
cols_to_check = [col for col in duplicate_names_df.columns if col not in exclude_cols]

i = 0
for game_name in duplicate_names_df['name'].unique():
    game_group = duplicate_names_df[duplicate_names_df['name'] == game_name]
    differing_cols = []
    for col in cols_to_check:
        if len(game_group[col].unique()) > 1:
            differing_cols.append(col)
    if'first_release_date' not in differing_cols:  # Print every 10th game to avoid too much output
        #print(f"{game_name}: {differing_cols}")
        pass
    i+=1

There are many duplicates but they may functionally differ so we'll keep them for now

## Conflicts in game modes columns

There are several game mode columns that seem to imply multiplayer functionality (such as Co-operative or Massively Multiplayer Online), but we should check if they actually do inherently imply multiplayer compatibility. If not, we should make sure we understand what the column is referring to.

In [63]:
coop_without_multiplayer = games_df.groupby(['game_modes_Co-operative', 'game_modes_Multiplayer'])[['name', 'first_release_date']].agg(lambda x: x.sample(1))
coop_without_multiplayer

name first_release_date
game_modes_Co-operative game_modes_Multiplayer                                
0                       0                       Auto Racing         2024-12-04
                        1                        M00m World         2025-11-06
1                       0                       Stream Town         2006-10-17
                        1                        Raiden III         2009-04-21

For co-op, it seems like an error in entry; games with coop should have a 1 in the multiplayer column.

In [64]:
mmo_without_multiplayer = games_df.groupby(['game_modes_Massively Multiplayer Online (MMO)', 'game_modes_Multiplayer'])[['name', 'first_release_date']].agg(lambda x: x.sample(1))
mmo_without_multiplayer

name  \
game_modes_Massively Multiplayer Online (MMO) game_modes_Multiplayer                                  
0                                             0                       Kid Mystic: Enchanted Edition   
                                              1                                     Rootin' Tootin'   
1                                             0                                     Blossom & Decay   
                                              1                                Nitro: Stream Racing   

                                                                     first_release_date  
game_modes_Massively Multiplayer Online (MMO) game_modes_Multiplayer                     
0                                             0                              1998-12-31  
                                              1                              2019-04-18  
1                                             0                              1997-01-09  
                                              1                              2017-02-18

Same for MMO

In [65]:
multiplayer_cols = ['Co-operative', 'Massively Multiplayer Online (MMO)']

for column in multiplayer_cols:
    rows_to_fix = games_df[(games_df['game_modes_' + column] > 0) & (games_df['game_modes_Multiplayer'] == 0)].index

    games_df.loc[rows_to_fix, 'game_modes_Multiplayer'] = 1

mmo_without_multiplayer = games_df.groupby(['game_modes_Massively Multiplayer Online (MMO)', 'game_modes_Multiplayer'])[['name', 'first_release_date']].agg(lambda x: x.sample(1))
mmo_without_multiplayer

name  \
game_modes_Massively Multiplayer Online (MMO) game_modes_Multiplayer                                  
0                                             0                                  Pokémon Fire Black   
                                              1                                         Recall News   
1                                             1                       Leviathan: Streams of Legends   

                                                                     first_release_date  
game_modes_Massively Multiplayer Online (MMO) game_modes_Multiplayer                     
0                                             0                              2003-04-10  
                                              1                              1970-01-01  
1                                             1                              2021-10-08

# Exploring multiplayer modes data

In [ ]:
clean_modes_fp = Path(config['multiplayer_modes_folder'] + config['multiplayer_modes_clean_fp'])

modes_df = pd.read_json(clean_modes_fp, orient='records')

print("Contains " + str(len(modes_df)) + " rows")
modes_df.head()

Contains 24107 rows


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen
0,9953,92273,False,False,False,0,2,False,0,0,Game Boy Color,False
1,1832,7153,True,True,True,2,0,False,0,0,Xbox 360,False
2,7987,57887,False,False,True,2,2,False,0,0,Xbox One,False
3,7,46076,False,False,False,0,30,False,0,0,PC (Microsoft Windows),False
4,10207,31256,False,False,False,0,0,False,0,0,Web browser,False


In [ ]:
modes_df.isna().mean().sort_values(ascending=False)

id                0.0
game              0.0
dropin            0.0
campaigncoop      0.0
offlinecoop       0.0
offlinecoopmax    0.0
offlinemax        0.0
onlinecoop        0.0
onlinecoopmax     0.0
onlinemax         0.0
platform          0.0
splitscreen       0.0
dtype: float64

-1 as a fill value works fine for all of these

In [ ]:
def isunique(df, subset):
    return len(df[subset].unique()) == len(df)

print("No duplicate IDS") if isunique(modes_df, 'id') else print("Duplicate IDs found")
print("No duplicate game ids") if isunique(modes_df, 'game') else print("Duplicate game ids found")

No duplicate IDS
Duplicate game ids found


In [ ]:
duplicate_game_ids_df = modes_df[modes_df.duplicated(subset=['game'], keep=False)].sort_values('game')
duplicate_game_ids_df = duplicate_game_ids_df.merge(games_df[['id', 'name']], left_on='game', right_on='id', how='left', suffixes=('_mode', '_game'))
duplicate_game_ids_df.head(10)

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,id_game,name
0,11592,72,False,False,True,2,0,True,2,0,PlayStation 3,True,72.0,Portal 2
1,11593,72,False,False,True,2,0,True,2,0,PC (Microsoft Windows),True,72.0,Portal 2
2,11594,72,False,False,True,2,0,True,2,0,Mac,True,72.0,Portal 2
3,11591,72,False,False,True,2,0,True,2,0,Xbox 360,True,72.0,Portal 2
4,11595,72,False,False,True,2,0,True,2,0,Linux,True,72.0,Portal 2
5,17435,83,False,False,False,0,0,False,0,0,PC (Microsoft Windows),True,83.0,Baldur's Gate: Dark Alliance
6,1631,83,False,True,True,2,0,False,0,0,PlayStation 2,False,83.0,Baldur's Gate: Dark Alliance
7,11138,121,True,True,False,0,0,False,0,0,Mac,False,121.0,Minecraft: Java Edition
8,11137,121,True,True,False,0,0,False,0,0,PC (Microsoft Windows),False,121.0,Minecraft: Java Edition
9,11139,121,True,True,False,0,0,False,0,0,Linux,False,121.0,Minecraft: Java Edition


Duplicate IDS often seems to indicate multi-platform releases

In [ ]:
full_df = modes_df.merge(games_df, left_on='game', right_on='id', how='left', suffixes=('', '_game'))
full_df.head()

,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,9953,92273,False,False,False,0,2,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1832,7153,True,True,True,2,0,False,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7987,57887,False,False,True,2,2,False,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7,46076,False,False,False,0,30,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10207,31256,False,False,False,0,0,False,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Handling conflicting values across columns

In [ ]:
columns = ['offlinecoop', 'offlinecoopmax', 'offlinemax']

def cap_at_1(x):
    return 1 if x >= 1 else x

for col in columns[1:]:
    full_df[col] = full_df[col].apply(cap_at_1)

full_df[columns[-1]].value_counts()

offlinemax
0    16645
1     7462
Name: count, dtype: int64

In [ ]:
#Sample one row from each unique combination of columns
selected_cols = ['name', 'first_release_date', 'platform'] + [col for col in full_df.columns if 'game_modes' in col]
conflict_scenarios_df = full_df.groupby(columns)[selected_cols].count()#agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  first_release_date  platform  \
offlinecoop offlinecoopmax offlinemax                                        
False       0              0           15022               15022     15059   
                           1            4755                4755      4758   
True        1              0            1584                1584      1586   
                           1            2703                2703      2704   

                                       game_modes_Battle Royale  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                              15022   
                           1                               4755   
True        1              0                               1584   
                           1                               2703   

                                       game_modes_Co-operative  \
offlinecoop offlinecoopmax offlinemax                            
False       0              0                             15022   
                           1                              4755   
True        1              0                              1584   
                           1                              2703   

                                       game_modes_Massively Multiplayer Online (MMO)  \
offlinecoop offlinecoopmax offlinemax                                                  
False       0              0                                                   15022   
                           1                                                    4755   
True        1              0                                                    1584   
                           1                                                    2703   

                                       game_modes_Multiplayer  \
offlinecoop offlinecoopmax offlinemax                           
False       0              0                            15022   
                           1                             4755   
True        1              0                             1584   
                           1                             2703   

                                       game_modes_Single player  \
offlinecoop offlinecoopmax offlinemax                             
False       0              0                              15022   
                           1                               4755   
True        1              0                               1584   
                           1                               2703   

                                       game_modes_Split screen  
offlinecoop offlinecoopmax offlinemax                           
False       0              0                             15022  
                           1                              4755  
True        1              0                              1584  
                           1                              2703

After inspecting some examples of each combination of the `offlinecoop`, `offlinecoopmax`, and `offlinemax` columns; the meanings can be interpreted as follows:

| offlinecoop | offlinecoopmax | offlinemax | Meaning |
| -------- | ------- | -------- | -------- |
| True | -1 | 1 | Misinput; trust data from games endpoint |
| True | 0 | 1 | Used the other column, fill coop max with max |
| True | 1 | -1 | Offline coop but no PVP |
| True | 1 | 0 | Offline coop but no PVP |
| True | 1 | 1 | Offline coop and PVP |
| False | -1 | -1 | No offline play |
| False | -1 | 0 | No offline play |
| False | -1 | 1 | Offline PVP but no coop |
| False | 0 | -1 | No offline play |
| False | 0 | 0 | No offline play |
| False | 0 | 1 | Inconsistent, assume no offline play |
| False | 1 | -1 | Inconsistent; assume no offline play|
| False | 1 | 0 | Inconsistent; assume no offline play|


There are only 5 error cases:

1. offlinecoop = True, offlinecoopmax \< 0, and offlinemax \> 0 

2. offlinecoop = True, offlinecoopmax = 0, and offlinemax \> 0 

3. offlinecoop = False, offlinecoopmax = 0, and offlinemax \> 0 

4. offlinecoop = False, offlinecoopmax \> 0, and offlinemax \< 0 

5. offlinecoop = False, offlinecoopmax \> 0, and offlinemax = 0 


In [ ]:
from data_cleaning_functions import get_replacement_function

offline = {'coop_column' : 'offlinecoop',
           'coop_max_column' : 'offlinecoopmax',
           'max_column' : 'offlinemax'}
online = {'coop_column' : 'onlinecoop',
           'coop_max_column' : 'onlinecoopmax',
           'max_column' : 'onlinemax'}

fixed_df = full_df.apply(get_replacement_function(**offline), axis = 1)
fixed_df = fixed_df.apply(get_replacement_function(**online), axis = 1)

conflict_scenarios_df = fixed_df.groupby(columns)[selected_cols].agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                          F-1 Race   
                         1              Little Racers Street   
True       1             0               Algos United: Live!   
                         1          Espionage: Mafia Evolved   

                                   first_release_date  \
onlinecoop onlinecoopmax onlinemax                      
False      0             0                 1988-12-31   
                         1                 1970-01-01   
True       1             0                 2024-06-04   
                         1                 2026-12-31   

                                                   platform  \
onlinecoop onlinecoopmax onlinemax                            
False      0             0          Sega Mega Drive/Genesis   
                         1           PC (Microsoft Windows)   
True       1             0           PC (Microsoft Windows)   
                         1           PC (Microsoft Windows)   

                                    game_modes_Battle Royale  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                               0.0   
                         1                               0.0   
True       1             0                               0.0   
                         1                               0.0   

                                    game_modes_Co-operative  \
onlinecoop onlinecoopmax onlinemax                            
False      0             0                              0.0   
                         1                              0.0   
True       1             0                              1.0   
                         1                              1.0   

                                    game_modes_Massively Multiplayer Online (MMO)  \
onlinecoop onlinecoopmax onlinemax                                                  
False      0             0                                                    0.0   
                         1                                                    0.0   
True       1             0                                                    0.0   
                         1                                                    0.0   

                                    game_modes_Multiplayer  \
onlinecoop onlinecoopmax onlinemax                           
False      0             0                             1.0   
                         1                             1.0   
True       1             0                             1.0   
                         1                             1.0   

                                    game_modes_Single player  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                               1.0   
                         1                               1.0   
True       1             0                               1.0   
                         1                               1.0   

                                    game_modes_Split screen  
onlinecoop onlinecoopmax onlinemax                           
False      0             0                              0.0  
                         1                              0.0  
True       1             0                              0.0  
                         1                              0.0